# 28 · Applications créatives — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch28_creatif/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch28_creatif/03_notebook.ipynb`, créée par `python tools/start_chapter.py 28`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 28.Q1 à 28.Q10, 28.R1 à 28.R3, 28.1, 28.2, 28.4 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | 28.11 à 28.15 | visualiser filtres et couches : un extracteur par hooks, la montée de gradient sur un réseau ImageNet, une galerie, la régularisation (*prochaine session de génération*) | | | |
| B | 28.16 et 28.17 | le deep dream sur une photo libre, couches, poids et masques (*prochaine session de génération*) | | | |
| C | 28.18 à 28.26 | le transfert de style : la matrice de Gram, des courbes, un 🐛, les reconstructions, le transfert complet, ses réglages, au-delà de Gram, une galerie documentée (*prochaine session de génération*) | | | |
| D | 28.27 à 28.29 | le texte mot à mot sur Verne, les stratégies de décodage, et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="28")  # your own library

## Objectifs et rappel express

- Extraire les activations d'un réseau pré-entraîné gelé avec des hooks, et visualiser ce que cherchent ses filtres et ses couches par montée de gradient sur l'image.
- Faire rêver un réseau sur une photo libre (deep dream).
- Écrire la matrice de Gram et les losses de contenu, de style et de variation totale, puis un transfert de style complet sous budget CPU, et documenter une galerie (sources, licences, réglages).
- Générer du texte mot à mot sur Verne et comparer les stratégies de décodage.

**Rappel express.** Montée de gradient : $\mathbf{x} \leftarrow \mathbf{x} + \eta\, \nabla_{\mathbf{x}}\, a(\mathbf{x})$, poids gelés. Gram : $G = \frac{1}{CHW} F F^\top$. Transfert de style : $L = \alpha \lVert F(\mathbf{x}) - F(\mathbf{c}) \rVert^2 + \beta \sum_\ell w_\ell \lVert G^\ell(\mathbf{x}) - G^\ell(\mathbf{s}) \rVert_F^2 + \lambda L_{TV}$. Température : $p_i \propto e^{z_i / T}$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 28.1, 28.2 et 28.4)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; plusieurs lettres : entre guillemets (`"ABD"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ 28.3 et 28.5, les estimations 🧮 28.6 et 28.7, la réflexion (🗣️ 28.8, ⚖️ 28.9, 📄 28.10) et l'entretien se corrigent avec `05_solutions.md`.

**28.Q1 — Pourquoi partir d'un réseau pré-entraîné comme VGG16**

In [ ]:
# 28.Q1: the values you found on paper
answer_28_Q1a = ...  # a) True or False
answer_28_Q1b = ...  # b) a whole number
answer_28_Q1c = ...  # c) the letter of your choice, e.g. "B"
answer_28_Q1d = ...  # d) the letter of your choice, e.g. "B"
answer_28_Q1e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_28_Q1a, answer_28_Q1b, answer_28_Q1c, answer_28_Q1d, answer_28_Q1e]):
    wb.check(f"28.Q1{letter}", answer)

**28.Q2 — Montée de gradient sur l'entrée, poids gelés**

In [ ]:
# 28.Q2: the values you found on paper
answer_28_Q2a = ...  # a) the letter of your choice, e.g. "B"
answer_28_Q2b = ...  # b) True or False
answer_28_Q2c = ...  # c) the letter of your choice, e.g. "B"
answer_28_Q2d = ...  # d) the letter of your choice, e.g. "B"
answer_28_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_28_Q2a, answer_28_Q2b, answer_28_Q2c, answer_28_Q2d, answer_28_Q2e]):
    wb.check(f"28.Q2{letter}", answer)

**28.Q3 — Visualiser une couche entière, pondérer ses filtres**

In [ ]:
# 28.Q3: the values you found on paper
answer_28_Q3a = ...  # a) the letter of your choice, e.g. "B"
answer_28_Q3b = ...  # b) the letter of your choice, e.g. "B"
answer_28_Q3c = ...  # c) True or False
answer_28_Q3d = ...  # d) a whole number
answer_28_Q3e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_28_Q3a, answer_28_Q3b, answer_28_Q3c, answer_28_Q3d, answer_28_Q3e]):
    wb.check(f"28.Q3{letter}", answer)

**28.Q4 — Deep dream : ce qui change par rapport à la visualisation d'une couche**

In [ ]:
# 28.Q4: the values you found on paper
answer_28_Q4a = ...  # a) the letters, e.g. "ABD"
answer_28_Q4b = ...  # b) True or False
answer_28_Q4c = ...  # c) the letter of your choice, e.g. "B"
answer_28_Q4d = ...  # d) the letter of your choice, e.g. "B"
answer_28_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_28_Q4a, answer_28_Q4b, answer_28_Q4c, answer_28_Q4d, answer_28_Q4e]):
    wb.check(f"28.Q4{letter}", answer)

**28.Q5 — Des yeux partout : l'écho des données d'entraînement**

In [ ]:
# 28.Q5: the values you found on paper
answer_28_Q5a = ...  # a) the letter of your choice, e.g. "B"
answer_28_Q5b = ...  # b) True or False
answer_28_Q5c = ...  # c) the letter of your choice, e.g. "B"
answer_28_Q5d = ...  # d) the letter of your choice, e.g. "B"
answer_28_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_28_Q5a, answer_28_Q5b, answer_28_Q5c, answer_28_Q5d, answer_28_Q5e]):
    wb.check(f"28.Q5{letter}", answer)

**28.Q6 — Image de base, référence de style, image générée : que minimise-t-on ?**

In [ ]:
# 28.Q6: the values you found on paper
answer_28_Q6a = ...  # a) the letter of your choice, e.g. "B"
answer_28_Q6b = ...  # b) the letter of your choice, e.g. "B"
answer_28_Q6c = ...  # c) the letter of your choice, e.g. "B"
answer_28_Q6d = ...  # d) True or False
answer_28_Q6e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_28_Q6a, answer_28_Q6b, answer_28_Q6c, answer_28_Q6d, answer_28_Q6e]):
    wb.check(f"28.Q6{letter}", answer)

**28.Q7 — Ce que la matrice de Gram retient et ce qu'elle oublie**

In [ ]:
# 28.Q7: the values you found on paper
answer_28_Q7a = ...  # a) the letter of your choice, e.g. "B"
answer_28_Q7b = ...  # b) the letter of your choice, e.g. "B"
answer_28_Q7c = ...  # c) True or False
answer_28_Q7d = ...  # d) the letter of your choice, e.g. "B"
answer_28_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_28_Q7a, answer_28_Q7b, answer_28_Q7c, answer_28_Q7d, answer_28_Q7e]):
    wb.check(f"28.Q7{letter}", answer)

**28.Q8 — Loss de contenu : couches précoces ou profondes ?**

In [ ]:
# 28.Q8: the values you found on paper
answer_28_Q8a = ...  # a) the letter of your choice, e.g. "B"
answer_28_Q8b = ...  # b) the letter of your choice, e.g. "B"
answer_28_Q8c = ...  # c) the letter of your choice, e.g. "B"
answer_28_Q8d = ...  # d) True or False
answer_28_Q8e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_28_Q8a, answer_28_Q8b, answer_28_Q8c, answer_28_Q8d, answer_28_Q8e]):
    wb.check(f"28.Q8{letter}", answer)

**28.Q9 — Loss de style, poids relatifs et variation totale**

In [ ]:
# 28.Q9: the values you found on paper
answer_28_Q9a = ...  # a) the letter of your choice, e.g. "B"
answer_28_Q9b = ...  # b) a whole number
answer_28_Q9c = ...  # c) the letter of your choice, e.g. "B"
answer_28_Q9d = ...  # d) True or False
answer_28_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_28_Q9a, answer_28_Q9b, answer_28_Q9c, answer_28_Q9d, answer_28_Q9e]):
    wb.check(f"28.Q9{letter}", answer)

**28.Q10 — Générer du texte mot à mot : ce qu'implique « le mot le plus probable »**

In [ ]:
# 28.Q10: the values you found on paper
answer_28_Q10a = ...  # a) the letter of your choice, e.g. "B"
answer_28_Q10b = ...  # b) True or False
answer_28_Q10c = ...  # c) the letter of your choice, e.g. "B"
answer_28_Q10d = ...  # d) True or False
answer_28_Q10e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_28_Q10a, answer_28_Q10b, answer_28_Q10c, answer_28_Q10d, answer_28_Q10e]):
    wb.check(f"28.Q10{letter}", answer)

**28.R1 — Un réseau qui sert de loss (le discriminateur)**

In [ ]:
# 28.R1: the values you found on paper
answer_28_R1a = ...  # a) the letter of your choice, e.g. "B"
answer_28_R1b = ...  # b) the letter of your choice, e.g. "B"
answer_28_R1c = ...  # c) True or False
answer_28_R1d = ...  # d) the letter of your choice, e.g. "B"
answer_28_R1e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_28_R1a, answer_28_R1b, answer_28_R1c, answer_28_R1d, answer_28_R1e]):
    wb.check(f"28.R1{letter}", answer)

**28.R2 — Mélange de contenu contre mélange de représentations**

In [ ]:
# 28.R2: the values you found on paper
answer_28_R2a = ...  # a) the letter of your choice, e.g. "B"
answer_28_R2b = ...  # b) the letter of your choice, e.g. "B"
answer_28_R2c = ...  # c) True or False
answer_28_R2d = ...  # d) a number
answer_28_R2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_28_R2a, answer_28_R2b, answer_28_R2c, answer_28_R2d, answer_28_R2e]):
    wb.check(f"28.R2{letter}", answer)

**28.R3 — Champ récepteur et hiérarchie des filtres de VGG16**

In [ ]:
# 28.R3: the values you found on paper
answer_28_R3a = ...  # a) a whole number
answer_28_R3b = ...  # b) a whole number
answer_28_R3c = ...  # c) True or False
answer_28_R3d = ...  # d) a whole number
answer_28_R3e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_28_R3a, answer_28_R3b, answer_28_R3c, answer_28_R3d, answer_28_R3e]):
    wb.check(f"28.R3{letter}", answer)

**Ex 28.1 — Tailles des activations et des matrices de Gram le long de VGG16**

In [ ]:
# Ex 28.1: the values you found on paper
answer_28_1a = ...  # a) a whole number
answer_28_1b = ...  # b) a whole number
answer_28_1c = ...  # c) a whole number
answer_28_1d = ...  # d) a whole number
answer_28_1e = ...  # e) a whole number
answer_28_1f = ...  # f) a whole number
answer_28_1g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_28_1a, answer_28_1b, answer_28_1c, answer_28_1d, answer_28_1e, answer_28_1f, answer_28_1g]):
    wb.check(f"28.1{letter}", answer)

**Ex 28.2 — Une matrice de Gram à la main**

In [ ]:
# Ex 28.2: the values you found on paper
answer_28_2a = ...  # a) a whole number
answer_28_2b = ...  # b) a whole number
answer_28_2c = ...  # c) a number
answer_28_2d = ...  # d) a number, 3 decimals
answer_28_2e = ...  # e) a number, 3 decimals
answer_28_2f = ...  # f) a number, 3 decimals
answer_28_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_28_2a, answer_28_2b, answer_28_2c, answer_28_2d, answer_28_2e, answer_28_2f, answer_28_2g]):
    wb.check(f"28.2{letter}", answer)

**Ex 28.4 — Losses de contenu, de style et de variation totale sur un mini-exemple**

In [ ]:
# Ex 28.4: the values you found on paper
answer_28_4a = ...  # a) a number
answer_28_4b = ...  # b) a number
answer_28_4c = ...  # c) a number
answer_28_4d = ...  # d) a number
answer_28_4e = ...  # e) a number
answer_28_4f = ...  # f) a number, 4 decimals
answer_28_4g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_28_4a, answer_28_4b, answer_28_4c, answer_28_4d, answer_28_4e, answer_28_4f, answer_28_4g]):
    wb.check(f"28.4{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire une montée de gradient sur l'image d'un réseau gelé, normalisation et plage des pixels comprises ?
2. Sais-tu calculer une matrice de Gram et dire ce qu'elle garde et ce qu'elle oublie ?
3. Sais-tu régler les trois poids d'un transfert de style en lisant la valeur de chaque terme ?
4. Sais-tu choisir une stratégie de décodage, et dire ce qu'est la perplexité ?

**Pour aller plus loin** : l'article de Gatys et coll. (2015) et l'article « Feature Visualization » de Distill (2017), cités dans la fiche. La suite : le ch. 29, les datasets et la préparation du projet final.

*Les exercices de code 28.11 à 28.29 (parties A à D : la visualisation, le deep dream, le transfert de style et la génération de texte) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 28` pour obtenir le notebook complet.*